# CPU 量化实训
理解损失、逐步误差补偿、低位宽权重与校准集。使用 CPU 与 NumPy。

在 `llm-quantization-lab` 仓库根目录执行 `python -m pip install -r requirements.txt`，然后打开本 Notebook。也可以直接运行 `lab.py`。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import lab
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("实验目录:", lab.ROOT)


## 1. 复核三轮梯度下降
课堂损失采用 `1/(2n)`。观察 a、b 与 loss，然后解释为什么第三轮尚未精确收敛。

In [ ]:
for row in lab.gradient_descent():
    print(row)
print(lab.toy_compensation())


## 2. 编码任务
用编辑器打开 `student_task.py`，补全三行 TODO；在终端运行 `python student_task.py`。检查通过后继续。

## 3. 运行位宽对照
同一 FP32 基线、同一独立测试集，对比 RTN、GPTQ-edu、AWQ-inspired。下面单元格等价于终端命令 `python lab.py --bits 8 4 2 --out results_bits`。

In [ ]:
from argparse import Namespace
args = Namespace(bits=[8,4,2], calibration=128, out="results_bits", retrain=False)
rows = lab.evaluate(args)
for r in rows:
    print(r["method"], r["bits"], round(r["accuracy"]*100,2), r["test_logit_mse"], r["actual_file_bytes"])


在浏览器打开 `results_bits/report.html`。把数据填写到“实验报告模板.txt”。

思考：在这个实验中，权重误差与 logits 误差的大小排序一定一致吗？4 bit 比 FP32 小多少倍？为什么不是严格 8 倍？

## 4. 校准样本对照
只改变校准样本数，不修改基线、位宽、测试集。RTN 的结果应该保持一致。

In [ ]:
cal16 = lab.evaluate(Namespace(bits=[4], calibration=16, out="cal16", retrain=False))
cal128 = lab.evaluate(Namespace(bits=[4], calibration=128, out="cal128", retrain=False))
for a,b in zip(cal16,cal128):
    print(a["method"], "logit MSE:", a["test_logit_mse"], "->", b["test_logit_mse"])


## 5. 可选：重新训练小模型
不是必做。训练只有两个小型线性层；`train_model` 使用交叉熵与 SGD，和前面的线性回归损失不同。下面先在内存训练，不覆盖仓库基线。

In [ ]:
RUN_OPTIONAL_TRAINING = False
if RUN_OPTIONAL_TRAINING:
    data = lab.load_data()
    my_model, history = lab.train_model(data, epochs=50)
    print("验证集准确率:", lab.accuracy(my_model, data["x_val"], data["y_val"]))


## 6. 提交与边界
提交报告、修改后的量化函数与三份 CSV。

本实验打包整数权重，但推理时解包为 FP32 并做 NumPy 矩阵乘；没有低比特 kernel。不能用这个实验声称真实 LLM 推理更快。GPTQ-edu 没有完整的分组/重排/块优化，AWQ-inspired 没有完整 clipping/mapping。数据来源与许可见数据来源与许可.txt。